# VZCrash on Google Colab (step M, phases 1 and 2): read-only look, then a small stratified subset

**Policy:** nothing from this dataset is downloaded to the laptop. This notebook reads the repository, saves two small text dumps and
one small parquet (about 25 MB) to Drive under `MyDrive/dhakaflow/vzcrash/`, and the laptop pulls back only the text files.

**The dataset.** `vzc-research-chapter/VZCrash`, revision `42e1482d005c0e545379d6cfa4939486c9fe4249`, licence **CC BY-NC 4.0** (non-commercial thesis use, with
attribution): Bianconcini, Monteagudo, Pjetri, Trinci, Taccari, "VZCrash: A Large-Scale IMU Dataset of Ego-Vehicle Crashes", IEEE ITSC 2026.
100 Hz tri-axial accelerometer in g (z measures 1 g at rest), gyroscope in deg/s, 1 Hz GPS speed in km/h, 16-second windows; labels
0 crash, 1 near_miss, 2 normal_driving (three-reviewer consensus with dashcam footage).

**Login.** Add your Hugging Face **read** token as a Colab secret named `HF_TOKEN` (key icon in the left bar). It is never printed.
The dataset is gated: accept its terms on its page first.

**Phase 1 (read-only):** list the files, save `README.md`, `explore.ipynb`, `pr_curve.ipynb` to Drive and dump their cells as text.
The repository's notebooks are **not run**; their loading and baseline logic will be reimplemented in our own script, credited.

**Phase 2:** a stratified subset of the **test** split only: 300 crashes and 300 negatives (150 near-misses; 150 normal-driving of which
75 are "kinematic distractors", the highest peak |acceleration| among the normal events read, and 75 random). Saved as one parquet on Drive.


In [ ]:
!pip -q install "huggingface_hub>=0.30" pyarrow
from google.colab import drive, userdata
drive.mount("/content/drive")
import os, json, hashlib, io, random, collections
OUT = "/content/drive/MyDrive/dhakaflow/vzcrash"; os.makedirs(OUT, exist_ok=True)
from huggingface_hub import HfApi, hf_hub_download, HfFileSystem, login
try:
    login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)        # from the Colab secret; never printed
except Exception as exc:
    print("no HF_TOKEN secret:", type(exc).__name__); login()                     # falls back to the interactive prompt
REPO, REV = "vzc-research-chapter/VZCrash", "42e1482d005c0e545379d6cfa4939486c9fe4249"
api = HfApi(); print("logged in as a user:", bool(api.whoami().get("name")))


## Phase 1: read-only

In [ ]:
info = api.dataset_info(REPO, revision=REV, files_metadata=True)
files = [{"path": s.rfilename, "bytes": s.size, "sha256": (s.lfs or {}).get("sha256") if isinstance(s.lfs, dict) else getattr(s.lfs, "sha256", None)} for s in sorted(info.siblings, key=lambda s: s.rfilename)]
json.dump({"repo": REPO, "revision": REV, "license": (info.card_data or {}).get("license"), "gated": info.gated, "files": files}, open(f"{OUT}/files.json", "w"), indent=1)
print(len(files), "files,", round(sum(f["bytes"] or 0 for f in files) / 1e9, 2), "GB in all")
for name in ("README.md", "explore.ipynb", "pr_curve.ipynb"):
    p = hf_hub_download(REPO, name, repo_type="dataset", revision=REV, local_dir=OUT)
    print("saved", name, round(os.path.getsize(p) / 1e3, 1), "KB")
    if name.endswith(".ipynb"):                                       # cells as text: code and markdown in full, outputs only when short text
        nb = json.load(open(p)); lines = [f"# {name} @ {REV}  ({len(nb['cells'])} cells)"]
        for i, c in enumerate(nb["cells"]):
            lines += [f"\n===== cell {i} [{c['cell_type']}] =====", "".join(c.get("source", []))]
            for o in c.get("outputs", []):
                txt = "".join(o.get("text", [])) or "".join((o.get("data") or {}).get("text/plain", []))
                if txt and len(txt) <= 2000: lines += ["----- output -----", txt]
                elif o.get("data", {}).get("image/png"): lines += ["----- output: an image (not copied) -----"]
        open(f"{OUT}/{name.replace('.ipynb', '_cells.txt')}", "w").write("\n".join(lines))
print("phase 1 done:", sorted(os.listdir(OUT)))


## Phase 2: the stratified subset (test split only, read row group by row group; the full files are never downloaded)

Selection rule, fixed before any result was seen: seed 7; 300 crashes (first 300 met in file order); 150 near-misses; normal driving:
read until a pool of 3 000 normal events, then 75 with the highest peak |a| (kinematic distractors) and 75 random from the rest.

In [ ]:
import pyarrow.parquet as pq, pyarrow as pa, numpy as np
fs = HfFileSystem()
test_files = sorted(f["path"] for f in files if f["path"].startswith("data/test-"))
print("test files:", test_files)
QUOTA = {"crash": 300, "near_miss": 150, "pool_normal": 3000}
keep = {"crash": [], "near_miss": []}; pool = []
rows_read = 0
def peak_g(gs): a = np.asarray(gs, dtype="float32"); return float(np.sqrt((a * a).sum(axis=1)).max()) if a.ndim == 2 and len(a) else float("nan")
for path in test_files:
    with fs.open(f"datasets/{REPO}@{REV}/{path}", "rb") as fh:
        pf = pq.ParquetFile(fh)
        for rg in range(pf.num_row_groups):
            t = pf.read_row_group(rg).to_pylist(); rows_read += len(t)
            for r in t:
                lab = {0: "crash", 1: "near_miss", 2: "normal"}[int(r["label"])]
                if lab == "crash" and len(keep["crash"]) < QUOTA["crash"]: keep["crash"].append(r)
                elif lab == "near_miss" and len(keep["near_miss"]) < QUOTA["near_miss"]: keep["near_miss"].append(r)
                elif lab == "normal" and len(pool) < QUOTA["pool_normal"]: pool.append(r)
            done = len(keep["crash"]) >= QUOTA["crash"] and len(keep["near_miss"]) >= QUOTA["near_miss"] and len(pool) >= QUOTA["pool_normal"]
            print(f"  {path} rg {rg}: read {rows_read}, crash {len(keep['crash'])}, near_miss {len(keep['near_miss'])}, normal pool {len(pool)}")
            if done: break
    if done: break
rng = random.Random(7)
pool.sort(key=lambda r: -peak_g(r["gsensor"]))
distractors, rest = pool[:75], pool[75:]
normal = distractors + rng.sample(rest, 75)
subset = keep["crash"] + keep["near_miss"] + normal
for r, g in zip(subset, ["crash"] * len(keep["crash"]) + ["near_miss"] * len(keep["near_miss"]) + ["normal_distractor"] * 75 + ["normal_random"] * 75): r["subset_group"] = g
table = pa.Table.from_pylist(subset)
out = f"{OUT}/subset_test.parquet"; pq.write_table(table, out)
sha = hashlib.sha256(open(out, "rb").read()).hexdigest()
man = {"repo": REPO, "revision": REV, "license": "CC BY-NC 4.0", "split": "test", "files_read": test_files, "rows_read": rows_read, "seed": 7, "rule": "first 300 crashes and 150 near-misses in file order; normal: pool of 3000, 75 highest peak |a| (distractors) + 75 random",
       "counts": collections.Counter(r["subset_group"] for r in subset), "parquet": "subset_test.parquet", "parquet_sha256": sha, "parquet_bytes": os.path.getsize(out),
       "peak_g": {g: [round(peak_g(r["gsensor"]), 2) for r in subset if r["subset_group"] == g][:5] for g in ("crash", "near_miss", "normal_distractor", "normal_random")}}
json.dump(man, open(f"{OUT}/subset_manifest.json", "w"), indent=1, default=int)
print(json.dumps(man, indent=1, default=int))


Done. Nothing to paste: the laptop pulls `files.json`, `explore_cells.txt`, `pr_curve_cells.txt` and `subset_manifest.json` with rclone.
The parquet stays on Drive; phase 3 (the paper's physical baseline, our WreckWatch rule, a gradient-boosting model) is a second notebook.